# argentina.geo.basemaps — Mapas argentinos con Folium

Recorrido **usando el paquete** para armar mapas interactivos HTML/Folium con fondo Argenmap (IGN) y toponimia argentina (incluido el rótulo correcto para las **Islas Malvinas**).

Cada celda genera un mapa Folium que se renderiza inline. Para abrirlos en navegador propio, usar `m.save('archivo.html')`.

## 1. Setup

In [1]:
import argentina as arg
from argentina.geo import basemaps, shapes
from argentina import provincias as provs
import folium

print(f"argentina v{arg.__version__}")
print(f"folium v{folium.__version__}")

argentina v0.0.14
folium v0.14.0


## 2. Mapa base de Argentina con Argenmap

Creamos un mapa sin tiles por defecto y le agregamos Argenmap del IGN.

In [2]:
m = folium.Map(
    location=[-38, -64],  # centro aproximado del país
    zoom_start=4,
    tiles=None,
)

basemaps.add_argenmap(m)
basemaps.add_creditos_argentina(m)
basemaps.add_layer_control(m)

m

## 3. Cargar y simplificar los polígonos del IGN

Las geometrías oficiales de `geo.shapes.provincias()` son muy detalladas (decenas de MB). Para mostrarlas a zoom país en Folium las simplificamos con `.simplify()` — sigue siendo perfectamente reconocible y el HTML del mapa queda liviano.

In [3]:
gdf = shapes.provincias()
# Tolerancia ~5 km en latitudes argentinas; suficiente para zoom país.
gdf['geometry'] = gdf.geometry.simplify(tolerance=0.05, preserve_topology=True)
gdf['region']   = gdf['in1'].apply(lambda c: provs.lookup(c).region)

tamano_aprox = gdf.to_json().__sizeof__() / 1024
print(f"{len(gdf)} provincias, GeoJSON simplificado ~ {tamano_aprox:,.0f} KB")

24 provincias, GeoJSON simplificado ~ 545 KB


/Users/tobiasyatche/miniconda3/lib/python3.10/site-packages/shapely/constructive.py:881: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


## 4. Provincias del IGN sobre fondo Argenmap

Combinamos `basemaps.add_argenmap` con los polígonos simplificados. Cada provincia se colorea según su región y tiene un tooltip.

In [4]:
colores_region = {
    'Pampeana':  '#ff9999',
    'Patagonia': '#99ccff',
    'NOA':       '#ffcc99',
    'NEA':       '#99ff99',
    'Cuyo':      '#cc99ff',
    'CABA':      '#ffff66',
}

def style_region(feature):
    color = colores_region.get(feature['properties']['region'], '#cccccc')
    return {
        'fillColor': color,
        'color': 'black',
        'weight': 0.7,
        'fillOpacity': 0.55,
    }

m = folium.Map(location=[-38, -64], zoom_start=4, tiles=None)
basemaps.add_argenmap(m)

folium.GeoJson(
    gdf[['in1', 'nam', 'region', 'geometry']],
    name='Provincias (IGN)',
    style_function=style_region,
    tooltip=folium.GeoJsonTooltip(
        fields=['nam', 'region', 'in1'],
        aliases=['Provincia:', 'Región:', 'Código INDEC:'],
    ),
).add_to(m)

basemaps.add_creditos_argentina(m)
basemaps.add_layer_control(m)
m

## 5. Argenmap vs OpenStreetMap

Activamos los dos fondos y dejamos el control de capas para alternar. Es la forma fácil de ver la diferencia de toponimia en Malvinas: el fondo argentino las muestra como **Islas Malvinas**, OSM las muestra como **Falkland Islands**.

In [5]:
m = folium.Map(location=[-38, -64], zoom_start=4, tiles=None)

basemaps.add_argenmap(m, name='Argenmap (IGN)')

folium.TileLayer(
    tiles='OpenStreetMap',
    name='OpenStreetMap',
    show=False,
).add_to(m)

basemaps.add_creditos_argentina(m)
basemaps.add_layer_control(m)
m

## 6. Zoom a Islas Malvinas

Vista cerrada sobre Malvinas con Argenmap. La toponimia es la oficial argentina.

In [6]:
m = folium.Map(
    location=[-51.75, -59.0],
    zoom_start=8,
    tiles=None,
)

basemaps.add_argenmap(m)
basemaps.add_creditos_argentina(m)
basemaps.add_layer_control(m)
m

## 7. Marcadores en las capitales provinciales

Para cada provincia tomamos el **representative_point** del polígono y mostramos un popup con metadata cruzada de `argentina.provincias`.

In [7]:
m = folium.Map(location=[-38, -64], zoom_start=4, tiles=None)
basemaps.add_argenmap(m)

for _, row in gdf.iterrows():
    centroide = row.geometry.representative_point()
    meta = provs.lookup(row['in1'])
    if meta is None:
        continue
    popup_html = (
        f"<b>{meta.nombre}</b><br>"
        f"Capital: {meta.capital}<br>"
        f"Región: {meta.region}<br>"
        f"ISO: {meta.iso_id} · INDEC: {meta.codigo_indec}"
    )
    folium.CircleMarker(
        location=[centroide.y, centroide.x],
        radius=4,
        color='black',
        weight=0.5,
        fill=True,
        fill_color='#cc0000',
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=meta.nombre,
    ).add_to(m)

basemaps.add_creditos_argentina(m)
basemaps.add_layer_control(m)
m

## 8. Guardar a HTML

Cualquier mapa Folium se puede serializar con `m.save(...)` y abrirlo desde el navegador.

In [8]:
from pathlib import Path

salida = Path('mapa_argentina_demo.html')
m.save(salida.as_posix())
print(f"Guardado: {salida.absolute()}  ({salida.stat().st_size // 1024} KB)")

Guardado: /Users/tobiasyatche/argentina/notebooks/mapa_argentina_demo.html  (34 KB)


## Notas

- Las etiquetas (Malvinas, etc.) viven **dentro del raster** de cada proveedor — el paquete no las puede reescribir desde el cliente. La estrategia es elegir un proveedor que ya rotule correctamente.
- Para overlays oficiales argentinos, sumar `argentina.geo.shapes` (provincias/departamentos del IGN) encima del fondo.
- `argentina[maps]` instala sólo `folium`. Para los polígonos del IGN hace falta también `argentina[geo]` (`geopandas`, `pyogrio`).
- Las geometrías del IGN son muy detalladas. Para mapas a zoom país conviene simplificar (`gdf.geometry.simplify(...)`) antes de embeberlas en un HTML Folium — el peso baja de decenas de MB a unos pocos KB.